---
title: "4.3 Multiple regression, categories and interactions"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/03-multiple-regression.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/04-regression/03-multiple-regression.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [4.2 Correlation and simple OLS](../04-regression/02-correlation-ols.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Add predictors and categories using an explicit formula.
- Interpret an adjusted coefficient and a reference category.
- Allow two groups to have different income slopes with an interaction.

## Why it matters

A **multiple regression** includes more than one predictor. Its slope describes a conditional fitted comparison: how the outcome varies with one predictor while the model's other predictors are held fixed. Adding controls does not automatically remove confounding—other causes associated with both predictor and outcome. We use made-up numbers to learn category syntax, then ask how the historical income–life expectancy association changes when region is included.

## Worked example

### Prepare the analysis libraries

Run this prepared cell **before importing other analysis libraries**. It uses the versions generated from the course's single dependency file. If needed, it installs free packages inside this temporary Python runtime, not on your own computer. An internet connection is needed for installation. If it reports that a restart is required, restart the Colab session and run from the top; mixing already loaded old code with newly installed libraries can produce misleading errors.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. In the pinned course environment this cell installs nothing. Colab may take a few minutes the first time. The current course environment uses Python 3.12; an older runtime that cannot install the pinned dependencies needs a current Python runtime rather than arbitrary version substitutions.

### Treat group labels as categories

`C(region)` tells the formula to treat region as a category, not a numeric scale. With an intercept and two groups, one group is the **reference category** and the other gets an indicator coefficient. This toy table uses made-up country-year values and invented broad groups; only its commands are being illustrated.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
small = pd.DataFrame({
    "country": ["Northland", "Northland", "Eastria", "Eastria", "Southport", "Southport"],
    "year": [2000, 2020, 2000, 2020, 2000, 2020],
    "gdp_pc": [8000, 15000, 21000, 30000, 2500, 4200],
    "life_exp": [68.0, 73.5, 76.0, 80.1, 58.2, 64.0],
})
small["region"] = ["Northern group", "Northern group", "Northern group", "Northern group", "Southern group", "Southern group"]
small["gdp_thousands"] = small["gdp_pc"] / 1000
toy = smf.ols("life_exp ~ gdp_thousands + C(region)", data=small).fit()
print(toy.params.round(3))

The group coefficient is an intercept difference relative to the reference group, at the same fitted GDP level. It is not a new country observation or a causal region effect. Arbitrary numeric region codes such as 1 and 2 would imply a numerical spacing unless treated as categories.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Define one common analysis sample

Keep the 2007 country cross-section and the predictors needed by all models in this lesson. Explicit complete-case preparation means the comparison will not accidentally change sample size when region is added. All these measurements are available in this teaching extract.

In [ ]:
countries = pd.read_csv(data_file)
sample = countries.loc[countries["year"] == 2007, ["country", "region", "life_exp", "gdp_pc"]].dropna().copy()
print("N:", len(sample))
print("All GDP values positive:", sample["gdp_pc"].gt(0).all())
sample["log_gdp_pc"] = np.log(sample["gdp_pc"])
print(sample["region"].value_counts())

There are 142 equally weighted country observations. Regions have unequal sample sizes, including only two countries in Oceania. A common slope with group intercepts is a modelling choice; it does not claim that each regional relationship is actually parallel.

### Add region intercept differences

Choose Africa as the reference explicitly with `Treatment(reference="Africa")`. This makes the meaning stable if row order changes. Fit a simple and a region-adjusted formula on the same observations, using the same HC1 uncertainty convention.

In [ ]:
simple = smf.ols("life_exp ~ log_gdp_pc", data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
adjusted_formula = 'life_exp ~ log_gdp_pc + C(region, Treatment(reference="Africa"))'
adjusted = smf.ols(adjusted_formula, data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
print(adjusted.params.round(3))
print("Simple income slope:", round(simple.params["log_gdp_pc"], 3))
print("Region-adjusted income slope:", round(adjusted.params["log_gdp_pc"], 3))

The income slope changes from about 7.203 to 4.631 years per log unit. The adjusted coefficient is the common within-region fitted slope under this additive specification, not a within-country time effect. Region coefficients compare fitted levels with Africa at a given income; they do not explain why those differences exist. Changing coefficients when adding region is evidence of sensitivity to specification, not proof that the adjusted coefficient is causal.

### Let a slope differ between two groups

An **interaction** allows one predictor's association to depend on another. `x * group` expands to x, group and x:group. Here the indicator `in_americas` distinguishes the source's Americas group from all other regions combined. Centre log income at 10000 source dollars: `log(gdp_pc / 10000)` is zero there, making the group intercept difference refer to a stated income level.

In [ ]:
sample["in_americas"] = (sample["region"] == "Americas").astype(int)
sample["log_gdp_centered"] = np.log(sample["gdp_pc"] / 10000)
interaction = smf.ols("life_exp ~ log_gdp_centered * in_americas", data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
print(interaction.params.round(3))
other_slope = interaction.params["log_gdp_centered"]
americas_slope = other_slope + interaction.params["log_gdp_centered:in_americas"]
print("Other regions' fitted slope:", round(other_slope, 3))
print("Americas fitted slope:", round(americas_slope, 3))

The main log-income coefficient is the slope when the indicator is 0. Add the interaction coefficient for the slope when it is 1. The group coefficient is a level difference at 10000 dollars, not a slope difference. “Other regions” combines heterogeneous countries, so this particular split is illustrative. An interaction's uncertainty requires the covariance between coefficients; adding their separate confidence-interval endpoints is not valid.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Change the reference category

Refit the region-adjusted model with Europe as reference. Compare its income coefficient with the Africa-reference fit.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
europe_reference = smf.ols('life_exp ~ log_gdp_pc + C(region, Treatment(reference="Europe"))', data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
print(europe_reference.params["log_gdp_pc"])
print(adjusted.params["log_gdp_pc"])

The income slopes agree up to numerical rounding. Changing a reference category re-expresses the same additive model; it changes intercept/indicator labels, not the fitted values or substantive comparison.

</details>

### 2. Translate an adjusted doubling

Use the region-adjusted common slope to express a doubling of GDP per person as a fitted life-expectancy difference.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
adjusted_doubling = adjusted.params["log_gdp_pc"] * np.log(2)
print(round(adjusted_doubling, 3))

The difference is about 3.210 years under the additive region specification. This is conditional model association, not the effect of an income intervention.

</details>

### 3. Read an interaction at its centre

Predict for two otherwise identical inputs at 10000 dollars, one outside and one inside the Americas. Compare their fitted difference with the group coefficient.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
prediction_inputs = pd.DataFrame({"log_gdp_centered": [0.0, 0.0], "in_americas": [0, 1]})
fitted_levels = interaction.predict(prediction_inputs)
centre_difference = fitted_levels.iloc[1] - fitted_levels.iloc[0]
print(centre_difference)
print(interaction.params["in_americas"])

The two displayed differences agree. At the centre the log-income term and interaction product are both zero, so the group intercept coefficient is exactly this model comparison.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A new prediction category was never fitted

A categorical predictor needs one of the fitted levels unless the modelling approach explicitly supports a new level.

In [ ]:
try:
    from patsy import PatsyError
    adjusted.predict(pd.DataFrame({"log_gdp_pc": [np.log(10000)], "region": ["Invented region"]}))
except PatsyError as error:
    print(f"{type(error).__name__}: {error}")

The error identifies an unexpected category. Check source coding and spelling; do not relabel a genuinely new group as the reference just to force a prediction.

### A formula ends with an incomplete addition

The formula grammar needs a term after each plus sign.

In [ ]:
try:
    smf.ols("life_exp ~ log_gdp_pc +", data=sample).fit()
except PatsyError as error:
    print(f"{type(error).__name__}: {error}")

Complete or remove the unfinished term. Formula symbols have their own meaning: * expands main terms and an interaction, while : requests only the interaction term.

## Recap

- Adjusted slopes depend on the specified controls, functional form and sample.
- A reference category defines the comparison for category coefficients.
- An interaction changes a slope; interpret main and interaction terms together.
- Controls and interactions do not by themselves identify causal effects.

**Next:** [4.4 Robust and clustered standard errors](../04-regression/04-robust-clustered.ipynb).

> **Coming from Stata / R / Excel?**
>
> C(region) is the role of a factor-variable declaration such as i.region in Stata. In R-style formulas, x * group likewise expands main effects and their interaction; do not interpret an interaction coefficient without its associated main effects.